# Multi-Agent CV ↔ Job Matcher
Upload CV PDFs in a Streamlit app, match them against jobs from a sheet and a live API, and automatically get a results sheet (with job links), a PDF report and a dashboard.

```
 Uploaded CV PDFs ────┐
 Jobs sheet (link/CSV)┼─► 1. Retrieval Agent
 Live jobs API ───────┘            │
                     ┌─────────────┴─────────────┐
                     ▼                           ▼
            2. CV Parser Agent (LLM)   3. Job Analyzer Agent (LLM)
                     └─────────────┬─────────────┘
                                   ▼
                        4. Matching Agent (code)
                                   ▼
                      5. Recommendation Agent (LLM)
                                   ▼
                        6. Summary Agent (LLM)
                                   ▼
        Actions: results sheet (with job links) · PDF report · live dashboard
```

**Why is matching plain code?** A small LLM is unreliable at scoring. The LLMs *understand* text (extract skills, write advice); code does the *scoring*, so the ranking is consistent and explainable.

## 1. Install

In [ ]:
!pip install -q transformers accelerate pypdf reportlab pandas requests streamlit

## 2. The agents and workflow (`agents.py`)
All agents, the automated actions, the live evaluation and the orchestrator (`run_pipeline`). The Qwen model loads when this file is imported.

In [ ]:
%%writefile agents.py
import json, re, time, os, requests
import pandas as pd
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModelForCausalLM
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from xml.sax.saxutils import escape

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # free; try Qwen2.5-3B-Instruct for better extraction

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")


# ---------------- LLM helpers ----------------
def ask_llm(prompt, max_new_tokens=300):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

stats = {"calls": 0, "failures": 0}      # used to measure reliability

def ask_json(prompt):
    """Ask the LLM for JSON. Returns a dict, or {} if the reply is not valid JSON."""
    stats["calls"] += 1
    try:
        return json.loads(re.search(r"\{.*\}", ask_llm(prompt), re.DOTALL).group())
    except Exception:
        stats["failures"] += 1
        return {}

def to_skills(value):
    """Turn the LLM's skills into a clean list of lowercase strings."""
    return [str(s).strip().lower() for s in value] if isinstance(value, list) else []

def same_skill(a, b):
    return a == b or (len(a) > 3 and len(b) > 3 and (a in b or b in a))

# ---- experience level: years of experience a job needs, from its title and description ----
TITLE_YEARS = {"intern": 0, "internship": 0, "trainee": 0, "graduate": 0, "entry": 0, "junior": 0, "jr": 0,
               "senior": 5, "sr": 5, "manager": 5, "lead": 6, "architect": 7, "principal": 8,
               "head": 8, "director": 8, "vp": 10, "chief": 10}
TOLERANCE = 1      # a job fits if it needs at most (your years + 1)

def to_years(value):
    """Turn the LLM's answer into a number of years (0 if unclear)."""
    try:
        return max(0.0, min(float(value), 50.0))
    except (TypeError, ValueError):
        found = re.search(r"\d+(\.\d+)?", str(value))
        return float(found.group()) if found else 0.0

def required_years(title, description):
    from_title = max([TITLE_YEARS.get(w, 0) for w in re.findall(r"[a-z]+", title.lower())], default=0)
    text = description.lower()
    found = re.findall(r"(\d{1,2})\s*\+?\s*(?:-\s*\d+\s*)?(?:years|yrs)\W+(?:of\s+)?(?:\w+\s+){0,3}experience", text)
    found += re.findall(r"experience\D{0,25}?(\d{1,2})\s*\+?\s*(?:-\s*\d+\s*)?(?:years|yrs)", text)
    from_text = min(int(n) for n in found) if found else 0
    return min(max(from_title, from_text), 15)


# ---------------- Agent 1: Retrieval (uploaded PDFs + jobs sheet + public API) ----------------
def load_jobs_sheet(sheet):
    """Read the jobs sheet (Google Sheet CSV link, CSV path or uploaded CSV)."""
    df = pd.read_csv(sheet).fillna("")
    df.columns = [c.strip().lower() for c in df.columns]
    missing = {"title", "company", "description"} - set(df.columns)
    if missing:
        raise ValueError(f"The jobs sheet is missing these columns: {', '.join(sorted(missing))}")
    link_col = next((c for c in ["link", "url", "job_link", "apply_link"] if c in df.columns), None)
    df["link"] = df[link_col].astype(str) if link_col else ""
    df["source"] = "sheet"
    return df[["title", "company", "description", "link", "source"]]


def retrieval_agent(cv_files, sheet, use_api, api_query):
    warnings, cvs = [], []
    for f in cv_files:
        name = os.path.basename(f.name)
        try:
            text = "\n".join(page.extract_text() or "" for page in PdfReader(f).pages)
        except Exception:
            warnings.append(f"{name}: could not be read as a PDF, skipped.")
            continue
        if not text.strip():
            warnings.append(f"{name}: no text found (scanned image?), skipped.")
            continue
        cvs.append({"file": name, "text": text})
    if not cvs:
        raise ValueError("None of the uploaded PDFs contained readable text.")

    jobs = []
    if sheet:
        jobs.append(load_jobs_sheet(sheet))
    if use_api:                                           # live jobs from a public API
        try:
            r = requests.get("https://remotive.com/api/remote-jobs",
                             params={"search": api_query, "limit": 5}, timeout=15).json()
            jobs.append(pd.DataFrame([{"title": j["title"], "company": j["company_name"],
                                       "link": j["url"], "source": "api",
                                       "description": re.sub(r"<[^>]+>", " ", j["description"])[:2000]}
                                      for j in r["jobs"]]))
        except Exception as e:
            warnings.append(f"Live jobs API unavailable: {e}")
    if not any(len(j) for j in jobs):
        raise ValueError("No jobs found. Add a jobs sheet or turn on the live jobs API.")
    return cvs, pd.concat(jobs, ignore_index=True), warnings


# ---------------- Agent 2: CV Parser (LLM) ----------------
def cv_parser_agent(cv):
    data = ask_json(f"""Extract information from this CV. Reply with JSON only, like:
{{"name": "Jane Doe", "skills": ["python", "sql"], "years_experience": 3}}
years_experience = years of full-time professional work only. Students, fresh graduates,
internships, courses and personal projects count as 0.

CV:
{cv['text'][:3000]}""")
    skills = to_skills(data.get("skills"))
    in_cv = sum(s in cv["text"].lower() for s in skills)           # skills that really appear in the CV
    return {"file": cv["file"], "name": str(data.get("name") or cv["file"]), "skills": skills,
            "years": to_years(data.get("years_experience", 0)),
            "grounded": round(100 * in_cv / len(skills)) if skills else 0}


# ---------------- Agent 3: Job Analyzer (LLM) ----------------
def job_analyzer_agent(job):
    data = ask_json(f"""Extract the required skills from this job post. Reply with JSON only, like:
{{"skills": ["python", "sql"]}}

Job title: {job['title']}
Description: {job['description'][:2000]}""")
    return {"title": job["title"], "company": job["company"], "source": job["source"],
            "link": str(job["link"]), "skills": to_skills(data.get("skills")),
            "years_needed": required_years(job["title"], job["description"])}


# ---------------- Agent 4: Matching (plain code, so the score is consistent) ----------------
def matching_agent(candidates, jobs):
    rows = []
    for c in candidates:
        for j in jobs:
            matched = [s for s in j["skills"] if any(same_skill(s, k) for k in c["skills"])]
            missing = [s for s in j["skills"] if s not in matched]
            score = round(100 * len(matched) / len(j["skills"])) if j["skills"] else 0
            fits = j["years_needed"] <= c["years"] + TOLERANCE          # experience level check
            rows.append({"file": c["file"], "candidate": c["name"], "candidate_years": round(c["years"], 1),
                         "job_title": j["title"], "company": j["company"], "source": j["source"],
                         "score": score, "fits": fits,
                         "experience_fit": "OK" if fits else f"Too senior (needs {j['years_needed']}+ yrs)",
                         "job_link": j["link"], "matched_skills": ", ".join(matched),
                         "missing_skills": ", ".join(missing)})
    df = pd.DataFrame(rows)
    # jobs that fit the experience level come first, then the best skill score
    df = df.sort_values(["file", "fits", "score"], ascending=[True, False, False]).reset_index(drop=True)
    df["rank"] = df.groupby("file").cumcount() + 1
    return df[["candidate", "candidate_years", "rank", "job_title", "company", "source", "score",
               "experience_fit", "job_link", "matched_skills", "missing_skills", "file"]]


# ---------------- Agent 5: Recommendation (LLM) ----------------
def recommendation_agent(candidate, top_matches):
    jobs = "\n".join(f"- {r.job_title} at {r.company}: {r.score}% skills match, experience: {r.experience_fit}, missing skills: {r.missing_skills or 'nothing'}"
                     for r in top_matches.itertuples())
    return ask_llm(f"""You are a career advisor.
Candidate: {candidate['name']} ({candidate['years']} years, skills: {', '.join(candidate['skills'])}).
Best job matches:
{jobs}

In 3 short sentences: say which job fits best and why, and which skills to learn to improve.
Only recommend jobs whose experience is OK. Mention a "Too senior" job only as a future goal.""", max_new_tokens=150)


# ---------------- Agent 6: Summary (LLM) ----------------
def summary_agent(matches):
    lines = "\n".join(f"- {r.candidate} ({r.candidate_years} years experience): best job {r.job_title} ({r.score}% skills, experience {r.experience_fit}), missing: {r.missing_skills or 'nothing'}"
                      for r in matches[matches["rank"] == 1].itertuples())
    return ask_llm(f"""Summarize these CV-to-job matching results in 4 short sentences for a hiring manager:
who fits best, and which skill gaps appear most often.

{lines}""", max_new_tokens=200)


# ---------------- Automated actions ----------------
def save_to_sheet(matches, path="results.csv"):       # CSV opens in Excel / Google Sheets
    matches.drop(columns="file").to_csv(path, index=False)


def generate_pdf_report(matches, summary, path="report.pdf"):
    styles = getSampleStyleSheet()
    story = [Paragraph("CV - Job Matching Report", styles["Title"]),
             Paragraph("<b>Summary:</b> " + escape(summary), styles["Normal"]), Spacer(1, 15)]
    for _, group in matches.groupby("file"):
        story.append(Paragraph(escape(group.candidate.iloc[0]), styles["Heading2"]))
        story.append(Paragraph("<b>AI advice:</b> " + escape(group.advice.iloc[0]), styles["Normal"]))
        for r in group.head(3).itertuples():
            line = escape(f"{r.rank}. {r.job_title} at {r.company} - {r.score}% skills, experience: {r.experience_fit} (missing: {r.missing_skills or 'none'})")
            if r.job_link:
                href = escape(r.job_link).replace('"', "&quot;")
                line += f' - <link href="{href}" color="blue">job link</link>'
            story.append(Paragraph(line, styles["Normal"]))
        story.append(Spacer(1, 15))
    SimpleDocTemplate(path).build(story)


# ---------------- Live evaluation (works on any uploaded CVs, no answer key needed) ----------------
def live_evaluation(candidates, matches, seconds):
    rows = []
    for c in candidates:
        best = int(matches[matches.file == c["file"]]["score"].max())
        ok = len(c["skills"]) >= 3 and c["grounded"] >= 80
        top3 = matches[(matches.file == c["file"]) & (matches["rank"] <= 3)]
        rows.append({"file": c["file"], "name": c["name"], "skills_found": len(c["skills"]),
                     "skills_really_in_cv_%": c["grounded"], "experience_years": round(c["years"], 1),
                     "top3_fit_experience": "yes" if (top3["experience_fit"] == "OK").all() else "no",
                     "best_match_%": best, "status": "OK" if ok else "CHECK"})
    valid = stats["calls"] - stats["failures"]
    return {"per_cv": pd.DataFrame(rows), "json_valid": f"{valid}/{stats['calls']}",
            "seconds": seconds, "seconds_per_cv": round(seconds / len(candidates), 1),
            "outputs": {f: os.path.exists(f) for f in ["results.csv", "report.pdf"]}}


# ---------------- Orchestrator ----------------
def run_pipeline(cv_files, sheet=None, use_api=True, api_query="data", years_override=None):
    if not cv_files:
        raise ValueError("Please upload at least one CV (PDF).")
    stats["calls"] = stats["failures"] = 0
    start = time.time()

    cvs, jobs, warnings = retrieval_agent(cv_files, sheet, use_api, api_query)        # 1. get the data
    candidates = [cv_parser_agent(cv) for cv in cvs]                                  # 2. understand CVs
    if years_override is not None:                                                    # user corrected the experience
        for c in candidates:
            c["years"] = float(years_override)
    job_list = [job_analyzer_agent(job) for job in jobs.to_dict("records")]           # 3. understand jobs
    matches = matching_agent(candidates, job_list)                                    # 4. score + rank

    advice = {c["file"]: recommendation_agent(c, matches[matches.file == c["file"]].head(3))
              for c in candidates}                                                    # 5. advice
    matches["advice"] = matches["file"].map(advice)
    summary = summary_agent(matches)                                                  # 6. overall summary

    save_to_sheet(matches)                                                            # action: sheet
    generate_pdf_report(matches, summary)                                             # action: report
    seconds = round(time.time() - start, 1)

    return matches, summary, live_evaluation(candidates, matches, seconds), warnings


## 3. The web app (`app.py`)
Streamlit interface: upload CVs, choose the jobs source, run, and see the results and the live evaluation.

In [ ]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="CV - Job Matcher", page_icon="🎯", layout="wide")
st.title("🎯 CV ↔ Job Matcher (Multi-Agent)")

with st.spinner("Loading the AI model (first time only)..."):
    import agents

# ---------------- Sidebar: inputs ----------------
with st.sidebar:
    st.header("1. Upload CVs")
    cv_files = st.file_uploader("CV PDFs (any number)", type="pdf", accept_multiple_files=True)

    st.header("2. Jobs")
    sheet_url = st.text_input("Google Sheet CSV link (optional)")
    sheet_file = st.file_uploader("...or upload a jobs CSV", type="csv")
    st.caption("Sheet columns: title, company, description. Optional: link (job page or application URL).")
    use_api = st.checkbox("Also fetch live jobs (Remotive API)", value=True)
    api_query = st.text_input("Live jobs keyword", "data")

    st.header("3. Your experience")
    levels = {"Auto-detect from the CV": None, "Fresh graduate (0 years)": 0, "Junior (1-2 years)": 2,
              "Mid-level (3-5 years)": 4, "Senior (6+ years)": 7}
    level = st.selectbox("Experience level", list(levels))
    st.caption("Use this if the detected experience is wrong. It applies to all uploaded CVs.")

    run = st.button("Run matching", type="primary")

if run:
    try:
        with st.spinner("The agents are working... this can take a few minutes."):
            sheet = sheet_file or sheet_url.strip() or None
            st.session_state["result"] = agents.run_pipeline(cv_files, sheet, use_api, api_query, levels[level])
    except Exception as e:
        st.error(str(e))

if "result" not in st.session_state:
    st.info("Upload your CV PDFs and choose a jobs source in the sidebar, then click **Run matching**.")
    st.stop()

matches, summary, evaluation, warnings = st.session_state["result"]
for w in warnings:
    st.warning(w)

tab_results, tab_eval = st.tabs(["Results", "Evaluation"])

# ---------------- Results tab ----------------
with tab_results:
    st.info(summary)
    candidate = st.selectbox("Candidate", sorted(matches["candidate"].unique()))
    top_n = st.slider("Number of top jobs", 1, 10, 5)
    hide_senior = st.checkbox("Hide jobs that are too senior for this CV", value=True)

    cand = matches[matches["candidate"] == candidate].sort_values("rank")
    st.caption(f"Detected experience: {cand['candidate_years'].iloc[0]} years. Jobs that need much more "
               "experience are ranked last and marked 'Too senior'.")
    df = cand[cand["experience_fit"] == "OK"] if hide_senior else cand
    df = df.head(top_n)

    if df.empty:
        st.warning("No job fits this CV's experience level. Untick the box above to see all jobs.")
    else:
        st.bar_chart(df.set_index("job_title")["score"])
        st.dataframe(df[["rank", "job_title", "company", "source", "score", "experience_fit", "job_link",
                         "matched_skills", "missing_skills"]],
                     hide_index=True, column_config={"job_link": st.column_config.LinkColumn("Job link")})
    st.markdown("**AI advice:** " + cand["advice"].iloc[0])

    c1, c2 = st.columns(2)
    c1.download_button("Download results sheet (CSV)", matches.drop(columns="file").to_csv(index=False),
                       "results.csv", "text/csv")
    c2.download_button("Download PDF report", open("report.pdf", "rb").read(), "report.pdf", "application/pdf")

# ---------------- Evaluation tab ----------------
with tab_eval:
    st.subheader("Live evaluation of this run")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Valid LLM JSON", evaluation["json_valid"])
    c2.metric("Total time", f"{evaluation['seconds']}s")
    c3.metric("Time per CV", f"{evaluation['seconds_per_cv']}s")
    c4.metric("Outputs created", f"{sum(evaluation['outputs'].values())}/{len(evaluation['outputs'])}")
    st.dataframe(evaluation["per_cv"], hide_index=True)
    st.caption("OK = at least 3 skills were extracted AND at least 80% of them literally appear in the CV text "
               "(so the model did not invent them). CHECK = review that CV manually.")


## 4. Testing & evaluation
Runs the pipeline on 4 test CVs with a known answer key. These test files are used only here, not by the app.

*This loads the model inside the notebook too; that is fine on a T4 GPU.*

In [ ]:
# These test CVs are ONLY used for testing here. The app itself uses the CVs you upload.
import os, pandas as pd, agents
from reportlab.pdfgen import canvas

os.makedirs("test_cvs", exist_ok=True)
test_cvs = {
    "sara_cv.pdf":    ("Sara Ahmed",   "Data Scientist",  3, "Python, SQL, machine learning, pandas, scikit-learn, statistics"),
    "omar_cv.pdf":    ("Omar Hassan",  "Web Developer",   2, "JavaScript, React, HTML, CSS, Node.js"),
    "lina_cv.pdf":    ("Lina Mostafa", "Data Analyst",    4, "Excel, SQL, Power BI, Tableau, statistics"),
    "youssef_cv.pdf": ("Youssef Ali",  "DevOps Engineer", 5, "Docker, Kubernetes, AWS, Linux, CI/CD"),
    "nour_cv.pdf":    ("Nour Samir",   "Fresh Graduate",  0, "Python, SQL, machine learning, pandas"),
}
for file, (name, title, years, skills) in test_cvs.items():
    pdf = canvas.Canvas(f"test_cvs/{file}")
    for i, line in enumerate([name, title, f"Experience: {years} years", f"Skills: {skills}"]):
        pdf.drawString(50, 800 - 25 * i, line)
    pdf.save()

pd.DataFrame([
    {"title": "Machine Learning Engineer",     "company": "DataCorp",   "link": "https://example.com/jobs/ml",     "description": "Python, machine learning, scikit-learn and SQL to build prediction models."},
    {"title": "Frontend Developer",            "company": "WebStudio",  "link": "https://example.com/jobs/web",    "description": "Build user interfaces with JavaScript, React, HTML and CSS."},
    {"title": "Business Intelligence Analyst", "company": "InsightCo",  "link": "https://example.com/jobs/bi",     "description": "Create dashboards and reports using SQL, Power BI, Excel and statistics."},
    {"title": "DevOps Engineer",               "company": "CloudWorks", "link": "https://example.com/jobs/devops", "description": "Manage infrastructure with Docker, Kubernetes, AWS and CI/CD pipelines on Linux."},
    {"title": "Senior Machine Learning Engineer", "company": "BigTech", "link": "https://example.com/jobs/senior-ml", "description": "5+ years of experience. Lead a team building models with Python, machine learning, SQL and pandas."},
    {"title": "Junior Data Scientist",         "company": "StartUp",    "link": "https://example.com/jobs/junior-ds", "description": "Entry level role using Python, SQL, pandas, machine learning and Tableau."},
]).to_csv("test_jobs.csv", index=False)

def open_cvs():
    return [open(f"test_cvs/{f}", "rb") for f in test_cvs]

files = open_cvs()
matches, summary, ev, warnings = agents.run_pipeline(files, "test_jobs.csv", use_api=False)
for f in files:
    f.close()

# 1. Accuracy: does each test CV get its intended job as the top match?
truth = {"sara_cv.pdf": "Machine Learning Engineer", "omar_cv.pdf": "Frontend Developer",
         "lina_cv.pdf": "Business Intelligence Analyst", "youssef_cv.pdf": "DevOps Engineer",
         "nour_cv.pdf": "Junior Data Scientist"}      # Nour is a fresh graduate: must NOT get the senior job
best = matches[matches["rank"] == 1]
correct = sum(truth[r.file] == r.job_title for r in best.itertuples())
print(f"Top-1 accuracy : {correct}/{len(best)}")

# 2. Reliability: how often did the LLM return valid JSON?
print(f"Valid LLM JSON : {ev['json_valid']}")

# 3. Efficiency
print(f"Total time     : {ev['seconds']}s for {len(test_cvs)} CVs ({ev['seconds_per_cv']}s per CV)")

# 4. Multi-step completion + job links in the results sheet
print("Outputs created:", ev["outputs"])
print("Job links saved:", matches["job_link"].str.startswith("http").all())

# 5. Experience level: a fresh graduate must not be recommended senior roles
nour = matches[matches.file == "nour_cv.pdf"]
print("Fresh graduate, top 3 all fit experience:", (nour[nour["rank"] <= 3]["experience_fit"] == "OK").all())
print(nour[["rank", "job_title", "score", "experience_fit"]].head(4).to_string(index=False))

# 6. Error handling
for label, args in [("no CVs", ([], "test_jobs.csv", False)), ("no jobs", (open_cvs(), None, False))]:
    try:
        agents.run_pipeline(*args)
    except ValueError as e:
        print(f"Handled error ({label}):", e)

# 7. The live evaluation table (the same one shown in the app)
ev["per_cv"]


## 5. Launch the app
Open the `trycloudflare.com` link printed by this cell.

In [ ]:
# 0. Stop any older app/tunnel that is still running (otherwise you keep seeing the old page)
!pkill -f streamlit
!pkill -f cloudflared

# 1. Install cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null

# 2. Launch the app in the background
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &>/content/streamlit_logs.txt &
!sleep 8

# 3. Create the public link (open the trycloudflare.com URL)
!cloudflared tunnel --url http://localhost:8501

## Documentation

### How to use the app
1. Upload any number of CV PDFs in the sidebar.
2. Add jobs: paste a Google Sheet CSV link, upload a CSV, and/or turn on the live jobs API.
3. If the detected experience is wrong, choose your level under **Your experience** (e.g. Fresh graduate).
4. Click **Run matching**. Results, the summary, download buttons and the live evaluation appear in two tabs.

### Jobs sheet format
Columns: `title`, `company`, `description`, and optionally `link` (the job page or application URL; `url`, `job_link` and `apply_link` also work). Live API jobs always include their link. The link is saved in the results sheet, the PDF report and shown as a clickable column in the dashboard.

### Experience-level check
Matching is not only about skills. Each job gets a *required years of experience* (from title words such as Senior, Lead, Head, Junior, and from phrases like "3+ years of experience" in the description). Each CV gets *years of professional experience* (the LLM is told that studies, internships and projects count as 0). A job **fits** if it needs at most (your years + 1). Jobs that fit are ranked first, jobs that are too senior are marked `Too senior (needs N+ yrs)` and ranked last, and the dashboard hides them by default. The advice and summary agents are also told to recommend only jobs whose experience fits.

### Objectives checklist
| Objective | Where it is done |
|---|---|
| 1. Data retrieval (PDFs, Sheets, APIs) | Retrieval Agent: uploaded CV PDFs + Google Sheet/CSV + Remotive API |
| 2. Processing & summarization with an LLM | CV Parser, Job Analyzer, Recommendation and Summary agents |
| 3. Automated actions | Results sheet (`results.csv`), PDF report, dashboard update |
| 4. Workflow orchestration | `run_pipeline()` runs the 6 agents in order, each output feeds the next |
| 5. Presentation & explanation | This documentation + `Video_Script.md` |
| 6. Testing & evaluation | Section 4 (accuracy, reliability, efficiency, completion, errors) + the live evaluation tab in the app |

### Evaluation
- **Test evaluation (section 4):** runs 5 test CVs with a known answer key (including a fresh graduate who must not get the senior job), so accuracy can be measured.
- **Live evaluation (Evaluation tab):** works on any CV you upload, with no answer key. It reports the valid-JSON rate, total time, time per CV, outputs created, and for each CV a check that the extracted skills really appear in the CV text (so the LLM did not invent them). It also shows the detected years of experience and whether all top-3 jobs fit that experience. CVs with fewer than 3 skills or under 80% grounded skills are marked CHECK.

### Limitations
- Small LLM (1.5B): skill extraction can miss or rename skills. Use a larger Qwen for better results.
- Scanned (image-only) PDFs have no text and are skipped with a warning.
- Matching uses skills and experience level only (not location, salary or education).
- Required experience is read from the job title and description with simple rules, so a job that never mentions seniority is treated as open to anyone.
- The test evaluation uses 4 made-up CVs, so it is a sanity check, not a benchmark.